# YOLOv8l 多数据集混合训练 — Kaggle 优化版
---
**模型**: YOLOv8l (44M), COCO 预训练  
**GPU**: Kaggle T4/P100 (16GB)  
**数据集**: archive + SH17 + SafetyVests + Mendeley

### 原代码问题修复
| 问题 | 原代码 | 修复 |
|------|--------|------|
| 模型加载 | `YOLO("yolov8l.yaml")` + `load()` | `YOLO("yolov8l.pt")` 一步到位 |
| copy_paste | 0.1 (分割增强, 无效) | 移除 |
| flipud | 0.2 (上下翻转) | 0.0 (人不倒立) |
| cos_lr | 缺失 | True |
| close_mosaic | 缺失 | 15 (最后15轮关闭) |
| amp | 缺失 | True (省显存+加速) |
| box/cls/dfl | 缺失 | 7.5/0.5/1.5 |
| multi_scale | True (慢) | False |
| patience | 12 (偏短) | 25 |
| seed | 缺失 | 42 |

In [ ]:
import torch, os, yaml
from pathlib import Path
from ultralytics import YOLO

# ── 环境 ──
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {gpu.total_memory/1e9:.1f} GB")

BASE = Path('/kaggle/working')
yaml_path = BASE / 'merged_dataset.yaml'  # 前一步生成的 YAML

## 训练参数（优化后）

In [ ]:
EPOCHS = 150
BATCH = 12
IMG_SIZE = 640

# ── 模型加载：一行搞定（结构 + COCO 权重）──
model = YOLO("yolov8l.pt")

# ── 训练 ──
results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMG_SIZE,
    device=0,
    workers=4,

    # ── 优化器 ──
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5,
    cos_lr=True,

    # ── 损失权重 ──
    box=7.5,
    cls=0.5,
    dfl=1.5,

    # ── 数据增强 (只保留有意义的) ──
    mosaic=0.8,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.0,          # 人不倒立，上下翻转无意义
    scale=0.5,
    translate=0.1,
    close_mosaic=15,     # 最后15轮关闭 mosaic，稳定收敛

    # ── 性能 ──
    amp=True,            # 混合精度，省显存 30% + 加速
    rect=False,

    # ── 早停 + 保存 ──
    patience=25,
    save=True,
    save_period=10,
    val=True,
    plots=True,

    # ── 其他 ──
    seed=42,
    pretrained=True,
    exist_ok=True,
    project=str(BASE / 'train_output'),
    name='yolov8l_ppe_multi',
)

print("✅ 训练完成")

## 评估 + 导出

In [ ]:
import shutil

# 验证
metrics = model.val()
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:     {metrics.box.mp:.4f}")
print(f"Recall:        {metrics.box.mr:.4f}")

# 逐个类别精度
if hasattr(metrics, 'ap_class_index'):
    names = ['person','helmet','vest','goggles','gloves','boots']
    for i, ap in enumerate(metrics.box.ap):
        print(f"  {names[i]:8s}: AP@0.5={metrics.box.ap50[i]:.4f}")

# 导出最佳模型
src = BASE / 'train_output/yolov8l_ppe_multi/weights/best.pt'
dst = BASE / 'best_yolov8l_multi.pt'
if src.exists():
    shutil.copy(src, dst)
    print(f"\n✅ 模型已导出: {dst} ({dst.stat().st_size/1e6:.1f} MB)")

## 原代码 vs 优化版对比

| 参数 | 原代码 | 优化版 | 原因 |
|------|--------|--------|------|
| 模型加载 | `YOLO(yaml) + load(pt)` ❌ | `YOLO("yolov8l.pt")` | 两步加载会丢失预训练权重 |
| epochs | 100 | 150 | 数据量大(32k), 需要更多轮 |
| copy_paste | 0.1 | 移除 | 实例分割增强, 对检测无效还费显存 |
| flipud | 0.2 | **0.0** | 安全帽/背心不会上下颠倒 |
| close_mosaic | 缺失 | **15** | 最后15轮关闭增强, 稳定收敛 |
| cos_lr | 缺失 | **True** | 余弦退火, 收敛更平滑 |
| amp | 缺失 | **True** | 省 30% 显存 + 1.5x 加速 |
| warmup | 3 | **5** | v8l 模型大, 预热要充分 |
| patience | 12 | **25** | 避免误停 (余弦退火后期可能波动) |
| box/cls/dfl | 缺失 | **7.5/0.5/1.5** | 明确的损失权重 |
| multi_scale | True | **False** | 对 640 固定输入帮助不大, 拖慢训练 |
| seed | 缺失 | **42** | 可复现 |